# 05.4 `Decimal` and `Fraction`

05.2 showed that floats cannot represent `0.1` exactly. This notebook covers the
two standard-library types that solve that problem in different ways.

## Theory

### The three numeric approaches

<table>
<tr><th>Type</th><th>Stores</th><th>Exact for</th><th>Speed</th></tr>
<tr><td><code>float</code></td><td>Binary fraction</td><td>Powers of two</td><td>Fastest</td></tr>
<tr><td><code>Decimal</code></td><td>Decimal digits</td><td>Any decimal</td><td>~100x slower</td></tr>
<tr><td><code>Fraction</code></td><td>Numerator/denominator</td><td>Any rational</td><td>Slowest</td></tr>
</table>

### `Decimal`: exact decimal arithmetic

`Decimal` stores numbers the way humans write them — as decimal digits with a
decimal exponent. So `Decimal("0.1")` is **exactly** one tenth.

This is what financial systems use. It implements the IBM General Decimal
Arithmetic specification, the same standard behind SQL's `DECIMAL` type.

**The critical rule:** construct from a **string**, never a float.

```python
Decimal("0.1")     # exactly 0.1
Decimal(0.1)       # inherits the float's error - pointless
```

### `Fraction`: exact rational arithmetic

`Fraction` stores an exact numerator and denominator, automatically reduced. So
`Fraction(1, 3)` is exactly one third — something neither `float` nor `Decimal`
can manage.

Use it when you need exact ratios, especially repeating ones.

### Choosing

- **Money, percentages, anything with decimal places that must be exact** →
  `Decimal`
- **Exact ratios, especially thirds and sevenths** → `Fraction`
- **Measurements, science, graphics, statistics** → `float`

In [ ]:
from decimal import Decimal

# The float problem, restated.
print("With float:")
print("   0.1 + 0.2       =", 0.1 + 0.2)
print("   == 0.3?         ", 0.1 + 0.2 == 0.3)

# Decimal, constructed from strings, is exact.
print("")
print("With Decimal:")
decimal_result = Decimal("0.1") + Decimal("0.2")
print("   Decimal('0.1') + Decimal('0.2') =", decimal_result)
print("   == Decimal('0.3')?              ", decimal_result == Decimal("0.3"))

# THE critical rule: never construct from a float.
print("")
print("Constructing from a float inherits the error:")
print("   Decimal(0.1)   ->", Decimal(0.1))
print("   Decimal('0.1') ->", Decimal("0.1"))
print("")
print("The first copied the float's approximation. Always use a string.")

In [ ]:
from decimal import Decimal

# Decimal preserves significant figures - it knows 1.10 has two decimal places.
print("Decimal remembers precision:")
print("   Decimal('1.1')  ->", Decimal("1.1"))
print("   Decimal('1.10') ->", Decimal("1.10"), "<- trailing zero kept")
print("   equal in value? ", Decimal("1.1") == Decimal("1.10"))

# Arithmetic keeps the appropriate number of places.
print("")
print("Arithmetic preserves scale:")
print("   Decimal('1.10') + Decimal('2.20') =", Decimal("1.10") + Decimal("2.20"))
print("   Decimal('1.5')  * Decimal('2')    =", Decimal("1.5") * Decimal("2"))

# Mixing with int is fine; mixing with float is not.
print("")
print("Mixing types:")
print("   Decimal('1.5') * 2    ->", Decimal("1.5") * 2, "<- int is fine")
try:
    Decimal("1.5") * 1.5
except TypeError as error:
    print("   Decimal('1.5') * 1.5  ->", type(error).__name__ + ":", error)
print("")
print("Python refuses to silently mix exact and approximate arithmetic.")

## Controlling precision and rounding

`Decimal` has a **context** holding the precision and rounding mode.

In [ ]:
from decimal import Decimal, getcontext, ROUND_HALF_UP, ROUND_DOWN, ROUND_UP

# The default context.
context = getcontext()
print("Default precision:", context.prec, "significant digits")
print("Default rounding: ", context.rounding)

# Division shows the precision limit.
print("")
print("One third at default precision:")
print("   Decimal(1) / Decimal(3) =", Decimal(1) / Decimal(3))

# Change the precision.
getcontext().prec = 50
print("")
print("At 50 digits:")
print("   Decimal(1) / Decimal(3) =", Decimal(1) / Decimal(3))

# Restore the default.
getcontext().prec = 28

# quantize() rounds to a fixed number of decimal places - essential for money.
print("")
print("Rounding to 2 decimal places with quantize:")
amount = Decimal("2.675")

for mode, label in [(ROUND_HALF_UP, "ROUND_HALF_UP"),
                    (ROUND_DOWN, "ROUND_DOWN"),
                    (ROUND_UP, "ROUND_UP")]:
    rounded = amount.quantize(Decimal("0.01"), rounding=mode)
    print(f"   {label:<15} {rounded}")

print("")
print("Compare with float, where 2.675 is already slightly below 2.675:")
print("   round(2.675, 2) ->", round(2.675, 2))

## A realistic money example

This is the case that matters. Small errors, repeated, become real money.

In [ ]:
from decimal import Decimal

# An invoice with three line items.
items = [
    ("Widget", "19.99", 3),
    ("Gadget", "5.49", 7),
    ("Doohickey", "0.10", 11),
]

TAX_RATE_FLOAT = 0.18
TAX_RATE_DECIMAL = Decimal("0.18")

# Calculate with floats.
float_subtotal = sum(float(price) * quantity for _, price, quantity in items)
float_tax = float_subtotal * TAX_RATE_FLOAT
float_total = float_subtotal + float_tax

# Calculate with Decimal.
decimal_subtotal = sum(Decimal(price) * quantity for _, price, quantity in items)
decimal_tax = decimal_subtotal * TAX_RATE_DECIMAL
decimal_total = decimal_subtotal + decimal_tax

print("Item          Price    Qty   Line total")
print("-" * 44)
for name, price, quantity in items:
    line = Decimal(price) * quantity
    print(f"{name:<13} {price:>6}   {quantity:>3}   {line:>10}")

print("")
print("                  float                    Decimal")
print("-" * 58)
print(f"subtotal   {float_subtotal:>20}   {decimal_subtotal:>20}")
print(f"tax        {float_tax:>20}   {decimal_tax:>20}")
print(f"total      {float_total:>20}   {decimal_total:>20}")

print("")
print("Rounded to 2 places for the customer:")
print("   float:  ", round(float_total, 2))
print("   Decimal:", decimal_total.quantize(Decimal("0.01")))
print("")
print("difference:", abs(float_total - float(decimal_total)))
print("")
print("Tiny here. Across millions of transactions it is a real problem,")
print("and auditors will find it.")

## `Fraction`: exact ratios

`Fraction` handles what neither float nor Decimal can: exact thirds.

In [ ]:
from fractions import Fraction
from decimal import Decimal

# One third, three ways.
print("One third:")
print("   float:    ", 1 / 3)
print("   Decimal:  ", Decimal(1) / Decimal(3))
print("   Fraction: ", Fraction(1, 3), "<- exact")

# The proof: multiply back by 3.
print("")
print("Multiplied by 3:")
print("   float:    ", (1 / 3) * 3, "== 1?", (1 / 3) * 3 == 1)
print("   Decimal:  ", (Decimal(1) / Decimal(3)) * 3)
print("   Fraction: ", Fraction(1, 3) * 3, "== 1?", Fraction(1, 3) * 3 == 1)

# Fractions reduce automatically.
print("")
print("Automatic reduction:")
print("   Fraction(6, 8)   ->", Fraction(6, 8))
print("   Fraction(100, 25)->", Fraction(100, 25))

# Several ways to construct.
print("")
print("Construction:")
print("   Fraction(3, 4)        ->", Fraction(3, 4))
print("   Fraction('3/4')       ->", Fraction("3/4"))
print("   Fraction('0.25')      ->", Fraction("0.25"))
print("   Fraction(Decimal('0.1')) ->", Fraction(Decimal("0.1")))
print("   Fraction(0.1)         ->", str(Fraction(0.1))[:30] + "...")
print("")
print("The last one exposes the float's true stored value - see 05.2.")

In [ ]:
from fractions import Fraction

# Exact arithmetic with repeating fractions.
third = Fraction(1, 3)
sixth = Fraction(1, 6)

print("Exact arithmetic:")
print("   1/3 + 1/6 =", third + sixth)
print("   1/3 - 1/6 =", third - sixth)
print("   1/3 * 1/6 =", third * sixth)
print("   1/3 / 1/6 =", third / sixth)

# Summing thirds is exact.
total = sum(Fraction(1, 3) for _ in range(3))
print("")
print("   1/3 + 1/3 + 1/3 =", total, "== 1?", total == 1)

# The float version is not.
float_total = sum(1 / 3 for _ in range(3))
print("   as floats:", float_total, "== 1?", float_total == 1)

# limit_denominator finds the best simple approximation.
import math
print("")
print("Approximating pi with limit_denominator:")
pi_fraction = Fraction(math.pi)
for limit in [10, 100, 1000, 1000000]:
    approximation = pi_fraction.limit_denominator(limit)
    error = abs(float(approximation) - math.pi)
    print(f"   denominator <= {limit:<8} {str(approximation):<16} error {error:.2e}")

print("")
print("22/7 appears at limit 10 - the classic school approximation.")

## Performance: exactness has a cost

In [ ]:
import time
from decimal import Decimal
from fractions import Fraction

def benchmark(label, make_value, repetitions=50000):
    """Time repeated addition for one numeric type."""
    left = make_value("1.1")
    right = make_value("2.2")

    start = time.perf_counter()
    for _ in range(repetitions):
        left + right
    elapsed = (time.perf_counter() - start) * 1000

    return label, elapsed


results = [
    benchmark("float", float),
    benchmark("Decimal", Decimal),
    benchmark("Fraction", lambda text: Fraction(text)),
]

baseline = results[0][1]

print("50,000 additions of 1.1 + 2.2:")
print("")
print("Type        Milliseconds   Relative")
print("-" * 40)
for label, elapsed in results:
    print(f"{label:<11} {elapsed:>10.2f}   {elapsed / baseline:>6.1f}x")

print("")
print("Decimal and Fraction are dramatically slower - and that is fine.")
print("Correctness matters more than speed for money. Just do not use")
print("them for numerical simulation or graphics.")

## Choosing between them

In [ ]:
guidance = [
    ("Money and currency", "Decimal", "exact decimal places, auditable"),
    ("Percentages and tax", "Decimal", "same reason"),
    ("Exact ratios (1/3)", "Fraction", "only type that can represent it"),
    ("Probability arithmetic", "Fraction", "exact odds and combinations"),
    ("Measurements", "float", "input is already approximate"),
    ("Scientific computing", "float", "speed matters, tiny error acceptable"),
    ("Graphics and geometry", "float", "speed matters enormously"),
    ("Statistics", "float", "or Decimal if reporting exact sums"),
    ("Counting things", "int", "exact and fast"),
]

print("Use case                  Type       Why")
print("-" * 70)
for use_case, choice, why in guidance:
    print(use_case.ljust(25), choice.ljust(10), why)

print("")
print("A note on storing money as integers:")
print("   Storing paise or cents as int is also correct and very fast.")
print("   Decimal is easier to read and handles tax rates more naturally.")
print("   Both are defensible. Floats are not.")

## Takeaways

1. `Decimal` stores **decimal digits**, so `Decimal("0.1")` is exactly one tenth.
2. **Always construct `Decimal` from a string** — `Decimal(0.1)` inherits the
   float's error.
3. `Decimal` preserves significant figures: `Decimal("1.10")` remembers the
   trailing zero.
4. Use `quantize()` with an explicit rounding mode to fix decimal places.
5. Python **refuses** to mix `Decimal` and `float` arithmetic — deliberately.
6. `Fraction` stores an exact numerator and denominator and auto-reduces, so
   `Fraction(1, 3) * 3 == 1` exactly.
7. `limit_denominator()` finds the best simple approximation — it produces `22/7`
   for pi.
8. Both are **far slower** than `float`. Use them where exactness matters, not
   everywhere.

## Try it yourself

1. Compare `Decimal(0.1)` with `Decimal("0.1")`. Explain the difference.
2. Sum `Decimal("0.01")` a hundred times. Is it exactly `1.00`?
3. Try `Decimal("1.5") + 1.5`. Why does Python refuse?
4. Use `quantize()` with each rounding mode on `2.5` and `3.5`.
5. Approximate `math.e` with `limit_denominator` at several limits.
6. Time the same calculation with all three types. Is the cost acceptable?